# 날짜 가이드 존 2차 실험: 가장자리 날짜를 가운데로 옮기면?

"상품을 가이드 존에 놓고 찍으면 날짜가 화면 가운데에 온다"를 **기존 사진으로** 흉내 낸다.

- **사진:** 화장품 개발용 300장 중 날짜가 가운데 40% 밖(경계 포함)에 있던 **141장** (143장 중 날짜가 사진에서 안 보이는 2장 제외)
  - 날짜 위치는 사진만 보고 표시했고, OCR 결과는 보지 않았다(`recenter_centers.txt`)
- **비교:** 같은 사진 × (원본 전체, **날짜를 가운데에 둔 60% 창**)
  - 창 크기는 모든 사진에서 가로·세로 60%로 고정하고 **위치만** 날짜 쪽으로 옮긴다. 날짜에 딱 맞게 자르지 않는다.
  - 날짜가 사진 끝에 붙어 있으면 창이 사진 밖으로 나갈 수 없어서 날짜가 완전히 가운데 오지는 않는다.
- **설정:** 지금 제출 파이프라인 그대로(OCR·Parser·재시도 같음).
- **순서:** 사진마다 원본/옮긴 창의 실행 순서를 무작위로 섞는다.
- **예상 시간:** 282건, 전원 연결 시 약 30분~1시간. 중간에 멈추면 4번 셀만 다시 실행한다(끝난 것은 건너뜀).
- **전원 연결 필수**(절전 모드면 시간이 4~5배 느려진다). 노트북 뚜껑 열어 두기. 주피터는 한 개만.

**끝나면** 5번 셀 결과 화면과 `recenter_experiment\recenter_ocr.jsonl`을 Claude에게 보낸다.

In [ ]:
# 1) 설정
from pathlib import Path
WORK_DIR  = r"C:\Users\user\itda_OCR"
COS_DIR   = WORK_DIR + r"\Cosmetics"
COS_LABEL = WORK_DIR + r"\cosmetics_labels_900001-900300.csv"
OUT_DIR   = WORK_DIR + r"\recenter_experiment"
MODE      = "full-stage"        # 제출 경로 시간은 따로 재고, 나머지 단계도 저장해 둔다(분석용)
IDS   = ["900001", "900006", "900007", "900009", "900012", "900013", "900015", "900016", "900017", "900018", "900019", "900021", "900023", "900025", "900027", "900029", "900030", "900038", "900039", "900041", "900042", "900045", "900050", "900052", "900053", "900056", "900062", "900063", "900064", "900065", "900067", "900068", "900069", "900070", "900071", "900072", "900075", "900076", "900077", "900080", "900081", "900082", "900086", "900091", "900092", "900093", "900095", "900096", "900098", "900099", "900100", "900101", "900103", "900104", "900105", "900106", "900107", "900108", "900109", "900111", "900112", "900114", "900116", "900119", "900122", "900124", "900125", "900127", "900128", "900132", "900135", "900137", "900139", "900140", "900143", "900145", "900146", "900152", "900153", "900154", "900158", "900159", "900160", "900164", "900165", "900166", "900168", "900171", "900176", "900179", "900186", "900187", "900188", "900189", "900190", "900194", "900197", "900200", "900201", "900207", "900209", "900211", "900216", "900217", "900218", "900224", "900225", "900227", "900228", "900230", "900231", "900234", "900235", "900237", "900244", "900246", "900247", "900249", "900252", "900256", "900257", "900259", "900261", "900262", "900263", "900264", "900266", "900268", "900273", "900275", "900278", "900279", "900280", "900282", "900283", "900285", "900286", "900289", "900290", "900293", "900294"]
CENTERS = {"900001": [0.42, 0.67], "900006": [0.45, 0.65], "900007": [0.45, 0.69], "900009": [0.4, 0.74], "900012": [0.35, 0.51], "900013": [0.72, 0.55], "900015": [0.66, 0.51], "900016": [0.83, 0.93], "900017": [0.45, 0.78], "900018": [0.47, 0.86], "900019": [0.53, 0.69], "900021": [0.4, 0.68], "900023": [0.68, 0.73], "900025": [0.48, 0.63], "900027": [0.48, 0.71], "900029": [0.53, 0.65], "900030": [0.66, 0.43], "900038": [0.61, 0.51], "900039": [0.47, 0.53], "900041": [0.35, 0.6], "900042": [0.37, 0.36], "900045": [0.34, 0.29], "900050": [0.4, 0.49], "900052": [0.55, 0.7], "900053": [0.46, 0.69], "900056": [0.61, 0.86], "900062": [0.58, 0.79], "900063": [0.5, 0.62], "900064": [0.8, 0.49], "900065": [0.42, 0.75], "900067": [0.63, 0.68], "900068": [0.58, 0.6], "900069": [0.58, 0.47], "900070": [0.62, 0.68], "900071": [0.52, 0.6], "900072": [0.77, 0.73], "900075": [0.6, 0.68], "900076": [0.55, 0.76], "900077": [0.58, 0.65], "900080": [0.35, 0.62], "900081": [0.54, 0.66], "900082": [0.63, 0.62], "900086": [0.63, 0.46], "900091": [0.48, 0.73], "900092": [0.37, 0.68], "900093": [0.6, 0.68], "900095": [0.61, 0.19], "900096": [0.45, 0.72], "900098": [0.58, 0.56], "900099": [0.75, 0.33], "900100": [0.5, 0.58], "900101": [0.43, 0.44], "900103": [0.64, 0.86], "900104": [0.36, 0.72], "900105": [0.34, 0.5], "900106": [0.57, 0.8], "900107": [0.58, 0.63], "900108": [0.5, 0.68], "900109": [0.47, 0.54], "900111": [0.62, 0.72], "900112": [0.58, 0.75], "900114": [0.4, 0.54], "900116": [0.47, 0.73], "900119": [0.65, 0.83], "900122": [0.7, 0.65], "900124": [0.43, 0.53], "900125": [0.43, 0.73], "900127": [0.36, 0.51], "900128": [0.66, 0.54], "900132": [0.25, 0.63], "900135": [0.62, 0.78], "900137": [0.54, 0.78], "900139": [0.62, 0.7], "900140": [0.58, 0.63], "900143": [0.53, 0.69], "900145": [0.47, 0.51], "900146": [0.45, 0.44], "900152": [0.62, 0.51], "900153": [0.52, 0.35], "900154": [0.37, 0.51], "900158": [0.5, 0.52], "900159": [0.5, 0.66], "900160": [0.47, 0.38], "900164": [0.47, 0.43], "900165": [0.5, 0.33], "900166": [0.4, 0.52], "900168": [0.5, 0.42], "900171": [0.47, 0.57], "900176": [0.45, 0.52], "900179": [0.47, 0.68], "900186": [0.35, 0.4], "900187": [0.47, 0.39], "900188": [0.33, 0.42], "900189": [0.25, 0.56], "900190": [0.4, 0.57], "900194": [0.5, 0.48], "900197": [0.43, 0.34], "900200": [0.5, 0.53], "900201": [0.35, 0.52], "900207": [0.38, 0.46], "900209": [0.6, 0.23], "900211": [0.52, 0.53], "900216": [0.45, 0.4], "900217": [0.47, 0.51], "900218": [0.5, 0.53], "900224": [0.47, 0.53], "900225": [0.65, 0.52], "900227": [0.65, 0.13], "900228": [0.75, 0.22], "900230": [0.65, 0.71], "900231": [0.55, 0.44], "900234": [0.6, 0.56], "900235": [0.37, 0.53], "900237": [0.7, 0.28], "900244": [0.47, 0.37], "900246": [0.67, 0.63], "900247": [0.43, 0.53], "900249": [0.37, 0.46], "900252": [0.4, 0.67], "900256": [0.52, 0.39], "900257": [0.39, 0.57], "900259": [0.35, 0.63], "900261": [0.57, 0.7], "900262": [0.27, 0.33], "900263": [0.42, 0.63], "900264": [0.52, 0.69], "900266": [0.52, 0.3], "900268": [0.5, 0.25], "900273": [0.42, 0.72], "900275": [0.52, 0.52], "900278": [0.5, 0.32], "900279": [0.52, 0.56], "900280": [0.55, 0.63], "900282": [0.66, 0.53], "900283": [0.57, 0.7], "900285": [0.42, 0.71], "900286": [0.55, 0.33], "900289": [0.4, 0.65], "900290": [0.6, 0.65], "900293": [0.52, 0.77], "900294": [0.52, 0.68]}   # 날짜 중심 (가로, 세로) 비율
VARIANTS = {"full": None, "recenter60": 0.60}
SEED = 20260929

In [ ]:
# 2) 환경 확인 + 자른 이미지 만들기 (OCR 없음)
import sys, os, json, time, random, csv
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
import importlib
import ocr_pipeline as op
import scripts.run_cosmetics_eval as ev
importlib.reload(ev)
from PIL import Image

missing = ev.weights_status()["missing"]
if missing:
    raise SystemExit("모델 파일이 없습니다: " + ", ".join(missing))
assert not any(900301 <= int(i) <= 900400 for i in IDS), "검증용 사진이 들어 있습니다"
files = {p.stem: p for p in Path(COS_DIR).iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
lost = [i for i in IDS if i not in files]
if lost:
    raise SystemExit("사진이 없습니다: " + ", ".join(lost))
with open(COS_LABEL, encoding="utf-8-sig", newline="") as f:
    LABELS = {r["image_id"]: r for r in csv.DictReader(f)}
print("코드 버전:", (Path(WORK_DIR) / "CODE_VERSION.txt").read_text(encoding="utf-8").splitlines()[0])

OUT = Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
PATHS = {}
for i in IDS:
    PATHS[(i, "full")] = files[i]
    rgb = op.decode_image(files[i])                 # 사진 방향(EXIF)을 바로잡은 뒤 자른다
    w, h = rgb.size
    for name, ratio in VARIANTS.items():
        if ratio is None:
            continue
        d = OUT / f"crops_{name}"; d.mkdir(exist_ok=True)
        path = d / f"{i}.png"
        if not path.exists():
            cx, cy = CENTERS[i]
            cw, ch = w * ratio, h * ratio            # 창 크기는 고정, 위치만 날짜 쪽으로
            x0 = min(max(cx * w - cw / 2, 0), w - cw); y0 = min(max(cy * h - ch / 2, 0), h - ch)
            rgb.crop((round(x0), round(y0), round(x0 + cw), round(y0 + ch))).save(path)   # PNG
        PATHS[(i, name)] = path
print(f"실행 대상: 원본 {len(IDS)} + 옮긴 60% 창 {len(IDS)} = {len(PATHS)}건")

In [ ]:
# 3) 모델 준비 + 예열(기록 안 함): 첫 실행이 느린 것이 결과에 섞이지 않게
t0 = time.perf_counter()
ENGINE = op.initialize_engine(**ev.ENGINE_OPTIONS)
print(f"모델 준비 {time.perf_counter() - t0:.0f}초")
t0 = time.perf_counter()
op.predict_image_detailed(ENGINE, files[IDS[0]])
print(f"예열 1장 {time.perf_counter() - t0:.1f}초 -> 전체 예상 약 {(time.perf_counter() - t0) * len(PATHS) * 1.5 / 60:.0f}분")

In [ ]:
# 4) 본 실행: 끝난 것은 건너뛴다. 멈추면 이 셀만 다시 실행.
import ctypes
DUMP = OUT / "recenter_ocr.jsonl"
def done():
    keys = set()
    if DUMP.exists():
        data = DUMP.read_bytes()
        if data and not data.endswith(b"\n"):
            data = data[:data.rfind(b"\n") + 1]; DUMP.write_bytes(data)
        for line in data.decode("utf-8").splitlines():
            if line.strip():
                r = json.loads(line); keys.add((r["image_id"], r["variant"]))
    return keys
def keep_awake(on):
    try:
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | (0x00000001 if on else 0))
    except Exception:
        pass
rng = random.Random(SEED)
plan = []
for i in IDS:
    names = [n for n in VARIANTS if (i, n) in PATHS]
    rng.shuffle(names)                       # 사진마다 실행 순서를 섞는다
    plan += [(i, n) for n in names]
keys = done(); todo = [p for p in plan if p not in keys]
print(f"남은 {len(todo)}건 / 전체 {len(plan)}건", flush=True)
keep_awake(True); started = time.perf_counter()
try:
    for k, (i, name) in enumerate(todo, 1):
        path = PATHS[(i, name)]
        with Image.open(path) as im:
            size = im.size
        rec = ev.process_image(ENGINE, Path(path), MODE)
        ocr_official = sum(s["seconds"] for s in rec["stages"] if s["official"])
        out = {"image_id": i, "variant": name, "size": list(size), "label": LABELS[i]["final_date"],
               "prediction": rec["prediction"]["final_date"], "method": rec["method"], "attempts": rec["attempts"],
               "official_seconds": rec["official_seconds"], "ocr_official_seconds": ocr_official,
               "extra_seconds": rec["extra_seconds"], "stages": rec["stages"]}
        with DUMP.open("a", encoding="utf-8", newline="\n") as f:
            f.write(json.dumps(out, ensure_ascii=False) + "\n"); f.flush(); os.fsync(f.fileno())
        if k % 5 == 0 or k == len(todo):
            el = time.perf_counter() - started
            print(f"[{k}/{len(todo)}] {i} {name} 경과 {el/60:.0f}분, 남은 예상 {el/k*(len(todo)-k)/60:.0f}분", flush=True)
finally:
    keep_awake(False)
print("끝:", DUMP)

In [ ]:
# 5) 간단 요약 (OCR 없음). 자세한 분석은 Claude가 한다.
import statistics, collections
from scripts.evaluation_common import normalize_truth
rows = [json.loads(l) for l in DUMP.read_text(encoding="utf-8").splitlines() if l.strip()]
truth = {i: normalize_truth(LABELS[i])["final_date"] for i in IDS}
by = collections.defaultdict(dict)
for r in rows:
    by[r["variant"]][r["image_id"]] = r
for name in VARIANTS:
    R = by.get(name, {})
    if not R:
        continue
    ok = sum(R[i]["prediction"] == truth[i] for i in R)
    none = sum(R[i]["prediction"] == "NONE" for i in R)
    t = [R[i]["official_seconds"] for i in R]
    print(f"{name:9s} {len(R):2d}장  정답 {ok} ({100*ok/len(R):.0f}%)  날짜 못 찾음(NONE) {none}  "
          f"1장 시간 평균 {statistics.mean(t):.1f}초 / 중앙값 {statistics.median(t):.1f}초")
for name in ("recenter60",):
    R = by.get(name, {}); F = by.get("full", {})
    both = [i for i in R if i in F]
    if both:
        a = sum(F[i]["prediction"] == truth[i] and R[i]["prediction"] != truth[i] for i in both)
        b = sum(F[i]["prediction"] != truth[i] and R[i]["prediction"] == truth[i] for i in both)
        print(f"같은 사진 비교 (원본 vs {name}, {len(both)}장): 원본만 맞음 {a}장, {name}만 맞음 {b}장")
print("\nClaude에게 보낼 파일:", DUMP)